<img src="https://www.10botics.com/images/logo_website_700x200-300x86.png" width="300"/>

# ✊✌️✋ 步驟 3: AI 影像辨識主程式

這個程式會啟動相機，將畫面即時傳送給我們訓練好的 AI 模型，並在畫面上顯示目前的辨識結果（剪刀、石頭、布或背景）。

## 1. 建立用於儲存模型的目錄
在上傳模型之前，我們需要先在 Raspberry Pi 裡面建立一個專用的資料夾。
我們可以使用 Linux 的指令 `mkdir` (意思就是 **M**ake **Dir**ectory 建立目錄) 來完成。

### 為什麼要加 `-p`？
我們會在指令中加入 `-p` 參數。這是一個保護機制，意思是：「如果這個資料夾已經存在了，就忽略它，不要顯示錯誤訊息」。這對於重複執行程式非常有幫助！

In [ ]:
# --- 練習 ---
# 請使用 !mkdir -p 指令來建立資料夾
# 資料夾名稱請設定為: teachable_machine_model

# 請在下方填寫你的指令：


### 💡 解答 (點擊展開)

In [ ]:
!mkdir -p teachable_machine_model

## 2. 上傳檔案 (Upload)
請看 JupyterLab 畫面的 **左側檔案總管**：

1.  **點兩下** 進入剛剛建立的 `teachable_machine_model` 資料夾。
2.  點擊上方選單的 **「上傳按鈕」** (圖示通常是一個 **向上箭頭** ⬆️)。
3.  選擇你電腦裡的兩個檔案：
    * `model.tflite`
    * `labels.txt`
4.  上傳完成後，確認左邊列表有這兩個檔案。

## 3. 匯入函式庫與設定路徑
首先，我們匯入必要的工具，並設定模型檔案的存放位置。

In [ ]:
pip install tflite_runtime

In [ ]:
import cv2
import numpy as np
import time
import os
# 專門用於執行 TFLite 模型的輕量級直譯器
from tflite_runtime.interpreter import Interpreter 
from picamera2 import Picamera2
from IPython.display import display, Image
import ipywidgets as widgets

# --- 練習：設定檔案路徑 ---
# 請將底線 ______ 修改為正確的檔案路徑字串

# 1. 設定模型檔案 (.tflite) 的路徑
MODEL_PATH = "________________________________________________________"

# 2. 設定標籤檔案 (.txt) 的路徑
LABEL_PATH = "________________________________________________________"

# --- 檢查路徑是否正確 (不要修改這裡) ---
if not os.path.exists(MODEL_PATH) or not os.path.exists(LABEL_PATH):
    print(f"❌ 錯誤：找不到檔案！")
    print(f"你設定的模型路徑: {MODEL_PATH}")
    print(f"你設定的標籤路徑: {LABEL_PATH}")
else:
    print("✅ 太棒了！檔案路徑確認無誤。")

### 💡 解答 (點擊展開)

In [ ]:
import cv2
import numpy as np
import time
import os
from tflite_runtime.interpreter import Interpreter 
from picamera2 import Picamera2
from IPython.display import display, Image
import ipywidgets as widgets

# 解答 1 ：完整絕對路徑
MODEL_PATH = "/home/pi/raspberry-pi-beginner/rock_paper_scissors/teachable_machine_model/model.tflite"
LABEL_PATH = "/home/pi/raspberry-pi-beginner/rock_paper_scissors/teachable_machine_model/labels.txt"

# 解答 2 ：相對路徑
# MODEL_PATH = "./teachable_machine_model/model.tflite"
# LABEL_PATH = "./teachable_machine_model/labels.txt"

# 檢查檔案是否存在
if not os.path.exists(MODEL_PATH) or not os.path.exists(LABEL_PATH):
    print("❌ 錯誤：找不到模型檔案！請檢查路徑是否正確。")
else:
    print("✅ 檔案路徑確認無誤。")

## 🧠 練習：文字魔術師 (String Split)

我們的 AI 模型回傳的標籤通常長這樣：`"0 Rock"`。
但在畫面上，我們只想要顯示英文單字（例如 `"Rock"`），不想要前面的數字。

**任務：** 請利用 `split()` 將文字切開，並取出我們要的部分。

**提示：**
1. 電腦從 0 開始數。
2. `"0 Rock"` 切開後會變成兩個東西：
   * 第 0 個是 `"0"`
   * 第 1 個是 `"Rock"` (這就是我們要的！)

In [ ]:
raw_label = "0 Rock"

# --- 練習：取出後面的英文單字 ---

# 第一步：使用 split() 切割文字
# parts = raw_label._______()

# 第二步：取出第 1 個項目 (記得電腦是從 0 開始數，所以第 2 個東西是 index 1)
# clean_label = parts[___]

print(f"處理後的文字: {clean_label}")

### 💡 解答 (點擊展開)

In [ ]:
raw_label = "0 Rock"

# 第一步：切開
parts = raw_label.split()
# 此時 parts 變成了 ['0', 'Rock']

# 第二步：拿第 1 個 (也就是 "Rock")
clean_label = parts[1]

print(f"處理後的文字: {clean_label}")

## 4. 載入模型與標籤
這段程式碼負責將 `model.tflite` 讀入記憶體，並準備好進行運算。我們也會讀取 `labels.txt`，這樣程式才知道 "0" 代表石頭、"1" 代表剪刀...

In [ ]:
def load_labels(path):
    """讀取標籤檔案，移除換行符號"""
    with open(path, 'r') as f:
        return [line.strip() for line in f.readlines()]

def load_model(path):
    """載入 TFLite 模型並分配張量 (Tensors)"""
    interpreter = Interpreter(model_path=path)
    interpreter.allocate_tensors()
    return interpreter

# 執行載入
try:
    labels = load_labels(LABEL_PATH)
    interpreter = load_model(MODEL_PATH)
    
    # 取得模型輸入和輸出的詳細資訊
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # 查看模型需要的輸入圖片大小
    height = input_details[0]['shape'][1]
    width = input_details[0]['shape'][2]
    
    print(f"✅ 模型載入成功！")
    print(f"   類別標籤: {labels}")
    print(f"   模型輸入尺寸: {width}x{height}")
    
except Exception as e:
    print(f"❌ 模型載入失敗: {e}")

## 5. 定義影像處理與推論函式
這是 AI 的核心邏輯。相機拍到的照片不能直接給 AI 吃，必須經過「預處理」：
1.  **縮放 (Resize)**：變成 224x224 像素。
2.  **擴充維度 (Expand Dims)**：變成 (1, 224, 224, 3) 的格式。
3.  **推論 (Inference)**：問 AI 這是什麼？

In [ ]:
def process_and_predict(image, interpreter, input_details, output_details):
    """
    1. 預處理影像
    2. 執行推論
    3. 回傳預測結果
    """
    # --- A. 預處理 ---
    # 取得模型需要的尺寸 (224x224)
    input_shape = input_details[0]['shape']
    target_height, target_width = input_shape[1], input_shape[2]
    
    # 縮放影像
    resized_image = cv2.resize(image, (target_width, target_height))
    
    # 檢查模型是否需要正規化 (float32) 或保持整數 (uint8)
    # 量化模型 (Quantized) 通常使用 uint8，不需要除以 255
    input_data = np.expand_dims(resized_image, axis=0)
    
    if input_details[0]['dtype'] == np.float32:
        # 如果是浮點數模型，通常需要正規化到 -1 ~ 1 之間
        input_data = (np.float32(input_data) - 127.5) / 127.5

    # --- B. 執行推論 (Inference) ---
    # 將處理好的圖片放入模型輸入端
    interpreter.set_tensor(input_details[0]['index'], input_data)
    
    # 開始計算
    interpreter.invoke()
    
    # --- C. 取得結果 ---
    # 從輸出端拿到預測數據 (一組信心指數，例如 [10, 240, 5, 0])
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]
    
    # 找出數值最大的那個索引 (例如 index 1 代表剪刀)
    max_index = np.argmax(output_data)
    
    # 計算信心百分比 (如果是 uint8，數值是 0-255，所以除以 255)
    confidence = output_data[max_index]
    if output_details[0]['dtype'] == np.uint8:
         confidence = confidence / 255.0
            
    return max_index, confidence

## 6. 主程式：啟動相機並開始辨識

In [ ]:
# 顯示影像的區域 (只保留影像 Widget)
image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

print("👇 程式執行中...")
print("如果要結束程式，請按上方工具列的 ⏹ (停止) 按鈕")

def main_loop():
    print("📷 相機啟動中...")
    
    try:
        # 使用 with 語法確保安全開關相機
        with Picamera2() as picam2:
            # 設定相機
            config = picam2.create_preview_configuration(main={"size": (640, 480)})
            picam2.configure(config)
            picam2.start()
            
            # 暖機
            time.sleep(2)
            print("🚀 AI 辨識開始！請對著鏡頭出拳！")

            # 改為無限迴圈，直到被手動中斷
            while True:
                # 1. 抓取影像
                frame = picam2.capture_array()
                
                # 2. 轉成 OpenCV 格式 (RGBA -> BGR) 並翻轉
                frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
                frame_flipped = cv2.flip(frame_bgr, 1) # 如果鏡頭倒置請用 -1，否則用 1 或 0
                
                # 3. 把圖片轉成 RGB (AI 需要 RGB)
                frame_rgb = cv2.cvtColor(frame_flipped, cv2.COLOR_BGR2RGB)
                
                # 4. 呼叫 AI 進行預測
                class_index, confidence = process_and_predict(frame_rgb, interpreter, input_details, output_details)
                
                # 取得預測名稱
                prediction_text = labels[class_index].split()[-1] 
                
                # 5. 將結果畫在螢幕上
                cv2.rectangle(frame_flipped, (0, 0), (250, 60), (0, 0, 0), -1)
                
                # 顯示類別名稱 (綠色字)
                cv2.putText(frame_flipped, f"{prediction_text}", (10, 35), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                
                # 顯示信心指數 (白色字)
                cv2.putText(frame_flipped, f"Conf: {confidence:.2f}", (10, 55), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

                # 6. 更新 Jupyter 畫面
                _, buffer = cv2.imencode('.jpeg', frame_flipped)
                image_widget.value = buffer.tobytes()
                
                # 稍微休息一下
                time.sleep(0.05)
                
    except KeyboardInterrupt:
        # 當你按下 Jupyter 的停止按鈕時，會執行這裡
        print("🛑 使用者手動停止程式")
        
    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
        
    finally:
        print("🛑 程式已結束，相機資源已釋放。")

# 執行主迴圈
main_loop()

<hr/>

## Congratulation! You have finished this chapter.

This jupyter notebook is created by 10Botics. <br>
For permission to use in school, please contact info@10botics.com <br>
All rights reserved. 2024.